# 05 · HAR-RV / Volatilidad Realizada — Supply Chain Adaptation

**Dataset real:** INEI — Índice de Precios al Consumidor (IPC) de Lima Metropolitana  
**Fuente:** Instituto Nacional de Estadística e Informática del Perú  
**URL directa:** https://m.inei.gob.pe/estadisticas/indice-tematico/price-indexes/  
**Descarga:** https://www.inei.gob.pe/media/MenuRecursivo/indices_tematicos/cuadro_001_ipc.xlsx

**Empresa ficticia:** Alicorp S.A.A. — categoría Aceites & Harinas, canal moderno Lima  
**Problema:** Los precios de insumos agrícolas (soya, trigo, maíz) importados tienen volatilidad de precios con memoria larga — afectan el costo de los productos terminados y por tanto la variabilidad de la demanda. El SS clásico usa σ histórica fija que no captura esta estructura de memoria múltiple.  
**Objetivo:** Construir la Varianza Realizada de los errores de forecast de precios IPC-Alimentos, aplicar HAR-RV para forecast de varianza a 4 semanas, y calcular el Safety Stock dinámico con horizonte forward-looking.  
**KPI:** SS dinámico que sube cuando la volatilidad de precios de insumos sube (señal de alerta temprana) y baja en períodos estables — sin intervención manual.

---

## Marco teórico — adaptación a Supply Chain

Sin datos intraperiodo, construimos la RV como varianza de errores de forecast a múltiples escalas:

$$RV_t^{(d)} = \epsilon_t^2 \qquad RV_t^{(w)} = \frac{1}{4}\sum_{i=0}^{3}\epsilon_{t-i}^2 \qquad RV_t^{(m)} = \frac{1}{13}\sum_{i=0}^{12}\epsilon_{t-i}^2$$

El HAR-RV sobre estos componentes:

$$\log(RV_{t+1}) = \beta_0 + \beta_d\log(RV_t^{(d)}) + \beta_w\log(RV_t^{(w)}) + \beta_m\log(RV_t^{(m)}) + \epsilon_{t+1}$$

Safety Stock forward-looking con forecast a $L$ semanas:

$$SS_t = z \cdot \sqrt{\widehat{RV}_{t+L}} \cdot \sqrt{L}$$

**Conexión IPC → Supply Chain:** la volatilidad del IPC-Alimentos de Lima predice la variabilidad de costos de insumos de Alicorp con un rezago de 2-4 semanas (transmisión del precio mayorista al precio industrial). La memoria larga del IPC justifica usar HAR en vez de GARCH.

**Referencias:** INEI (2024). Metodología IPC Lima Metropolitana. Corsi (2009). *JFEC* 7(2).

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm
warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)

C = dict(
    ipc='#2563EB',  rv='#DC2626',    har='#15803D',
    garch='#F59E0B', fill='#FEE2E2', neutral='#94A3B8',
    ss_dyn='#DC2626', ss_cls='#94A3B8',
    food='#F59E0B',  energy='#7C3AED'
)
np.random.seed(42)
print('✓ OK')

In [ ]:
# ── DATOS — INEI IPC Lima Metropolitana ──────────────────────────────────────
# Fuente: INEI — Índice de Precios al Consumidor, Lima Metropolitana
# Serie mensual publicada: base 2009=100
# Categorías disponibles: General, Alimentos y Bebidas, Energía
#
# OPCIÓN A — Descarga real:
#   import requests
#   url = 'https://www.inei.gob.pe/media/MenuRecursivo/indices_tematicos/cuadro_001_ipc.xlsx'
#   df_raw = pd.read_excel(url, sheet_name=0, skiprows=3, engine='openpyxl')
#
# OPCIÓN B — Simulación calibrada con estadísticos reales INEI 2015-2024:
#   IPC General Lima: inflación media ~3.5% anual, pico COVID ~8.8% (2022)
#   IPC Alimentos:    inflación media ~4.2% anual, pico 2022 ~15.3%
#                     Volatilidad mensual σ ≈ 0.6%  (normal) → 2.1% (crisis)
#   Clusters documentados:
#     - Sequía El Niño 2016-17: alza alimentos frescos
#     - COVID disruption 2020: alza aceites, harinas
#     - Guerra Ucrania 2022: alza aceite soya, trigo (máx histórico)
#     - Normalización 2023-24
#
# Convertimos a SEMANAL interpolando (INEI publica mensual, usamos interpolación
# spline para obtener serie semanal proxy — práctica estándar en investigación)

try:
    import urllib.request
    url = 'https://www.inei.gob.pe/media/MenuRecursivo/indices_tematicos/cuadro_001_ipc.xlsx'
    with urllib.request.urlopen(url, timeout=10) as r:
        import io
        content = r.read()
    df_raw = pd.read_excel(io.BytesIO(content), skiprows=3, engine='openpyxl')
    # Procesamiento básico (estructura puede variar)
    SOURCE = 'INEI (datos reales)'
    print(f'✓ Descargado desde INEI: {df_raw.shape}')
    USE_REAL = True
except Exception as e:
    print(f'INEI no disponible ({type(e).__name__}) — usando simulación calibrada')
    SOURCE = 'Simulación calibrada (estadísticos reales INEI 2015-2024)'
    USE_REAL = False

if not USE_REAL:
    # ── Generar serie mensual IPC Alimentos calibrada ────────────────────────
    # 120 meses (10 años: Ene 2015 – Dic 2024)
    months = pd.date_range('2015-01-01', periods=120, freq='MS')

    # Inflación mensual base + eventos
    inf_base  = 0.0035   # ~4.2% anual
    inf_shocks = np.random.normal(0, 0.006, 120)  # σ mensual normal

    # Eventos documentados
    events = [
        (24, 30, 0.012),   # El Niño 2016-17: alimentos frescos
        (60, 66, 0.018),   # COVID 2020: disruption supply
        (84, 96, 0.021),   # Guerra Ucrania 2022: soya, trigo, aceite
    ]
    for start, end, sigma_extra in events:
        inf_shocks[start:end] += np.random.normal(0.008, sigma_extra, end-start)

    inf_monthly = inf_base + inf_shocks
    ipc_monthly = 100 * np.cumprod(1 + inf_monthly)
    ipc_food    = pd.Series(ipc_monthly, index=months, name='IPC_Alimentos')

    # Interpolar a semanal (spline)
    weeks = pd.date_range('2015-01-05', periods=520, freq='W-MON')
    ipc_interp = ipc_food.reindex(ipc_food.index.union(weeks))\
                         .interpolate(method='cubic')\
                         .reindex(weeks)
    ipc_weekly = ipc_interp.dropna()

    # Variación semanal (tasa de cambio)
    chg_weekly = ipc_weekly.pct_change().dropna()

    print(f'\nFuente : {SOURCE}')
    print(f'Período: {chg_weekly.index[0].date()} → {chg_weekly.index[-1].date()}')
    print(f'n      : {len(chg_weekly)} semanas')
    print(f'Var. μ : {chg_weekly.mean():.5f} ({chg_weekly.mean()*52:.2%} anual)')
    print(f'Var. σ : {chg_weekly.std():.5f} ({chg_weekly.std()*np.sqrt(52):.2%} anual)')

## Mini-EDA

In [ ]:
# ── EDA 1/2 — Estadísticos clave ─────────────────────────────────────────────
s = chg_weekly
print(f'{"Métrica":<28} {"Valor":<16} Nota')
print('─' * 72)
rows = [
    ('n semanas',             len(s),                          '10 años'),
    ('Var. media semanal',    f'{s.mean():.5f}',               f'{s.mean()*52:.2%} inflación anual IPC-Ali'),
    ('Volatilidad semanal σ', f'{s.std():.5f}',                f'{s.std()*np.sqrt(52):.2%} anual'),
    ('Skewness',              f'{s.skew():.3f}',               '> 0 → alzas más extremas que bajas'),
    ('Kurtosis',              f'{s.kurt():.3f}',               '> 0 → fat tails'),
    ('p01 / p99',             f'{s.quantile(.01):.5f} / {s.quantile(.99):.5f}', 'rango extremo'),
    ('Máx variación semana',  f'{s.max():.4%}',                'pico de crisis'),
    ('Mín variación semana',  f'{s.min():.4%}',                ''),
]
for label, val, note in rows:
    print(f'{label:<28} {str(val):<16} {note}')

# ACF rápida de s²
from statsmodels.tsa.stattools import acf
acf_s2 = acf(s**2, nlags=20, fft=True)
ci = 1.96 / np.sqrt(len(s))
sig_lags = [i for i in range(1, 21) if abs(acf_s2[i]) > ci]
print(f'\nACF(variación²) significativa en rezagos: {sig_lags}')
print(f'→ {"Long memory confirmado → HAR justificado" if len(sig_lags) > 5 else "Memoria moderada"}')

In [ ]:
# ── EDA 2/2 — IPC + variación semanal + ACF ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(
    f'Mini-EDA — INEI IPC Alimentos & Bebidas · Lima Metropolitana\n{SOURCE}',
    fontsize=10, y=1.02
)

# IPC nivel
ax = axes[0]
ax.plot(ipc_weekly.index, ipc_weekly.values, color=C['ipc'], lw=1.0)
ax.set_title('IPC Alimentos (base 2009=100)', fontsize=9)
ax.set_ylabel('Índice')
ax.grid(axis='y', alpha=0.3)

# Variación semanal
ax2 = axes[1]
colors_s = np.where(s >= 0, C['rv'], C['ipc'])
ax2.bar(s.index, s.values * 100, color=colors_s, alpha=0.7, width=5)
ax2.axhline(0, color='black', lw=0.4)
ax2.set_title('Variación semanal IPC (%)', fontsize=9)
ax2.set_ylabel('%')
ax2.grid(axis='y', alpha=0.3)

# ACF de variación²
ax3 = axes[2]
lags_acf = range(1, 21)
ax3.bar(lags_acf, acf_s2[1:21], color=C['rv'], alpha=0.7, width=0.8)
ax3.axhline(ci,  color=C['neutral'], lw=1.0, ls='--')
ax3.axhline(-ci, color=C['neutral'], lw=1.0, ls='--')
ax3.axhline(0, color='black', lw=0.4)
ax3.set_title('ACF(variación²) — long memory', fontsize=9)
ax3.set_xlabel('Rezago (semanas)')
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/supply_eda.png', dpi=130, bbox_inches='tight')
plt.show()
print('✓ data/supply_eda.png')

In [ ]:
# ── CONSTRUIR RV Y VARIABLES HAR ──────────────────────────────────────────────
# RV semanal = variación² (proxy — sin datos intraperiodo)
# Escalas: d=1 sem, w=4 sem, m=13 sem (1 trimestre)

def build_supply_har(series, w=4, m=13):
    """
    Construye variables HAR para series de demanda/precio semanal.
    Sin datos intraperiodo: RV^(d) = ε² · w=4 sem · m=13 sem
    """
    eps2 = series**2
    df_out = pd.DataFrame(index=series.index)
    df_out['RV_d']    = eps2
    df_out['RV_w']    = eps2.rolling(w).mean()
    df_out['RV_m']    = eps2.rolling(m).mean()
    df_out['RV_next'] = eps2.shift(-1)
    # Log-transform
    for col in ['RV_d','RV_w','RV_m','RV_next']:
        df_out[f'log{col}'] = np.log(df_out[col].clip(lower=1e-12))
    return df_out.dropna()

har_df = build_supply_har(s)

print(f'Obs. disponibles para HAR: {len(har_df)}')
print('\nCorrelaciones entre escalas (multicolinealidad esperada):')
print(har_df[['RV_d','RV_w','RV_m']].corr().round(3).to_string())

In [ ]:
# ── ESTIMAR HAR-RV (log) Y GARCH BENCHMARK ───────────────────────────────────
from scipy.optimize import minimize

# Train/Test split: 80/20
split = int(len(har_df) * 0.80)
train, test = har_df.iloc[:split], har_df.iloc[split:]

def ols_fit(X, y):
    Xb = np.column_stack([np.ones(len(X)), X])
    b  = np.linalg.lstsq(Xb, y, rcond=None)[0]
    yhat = Xb @ b
    r2 = 1 - ((y-yhat)**2).sum() / ((y-y.mean())**2).sum()
    return b, r2

def ols_predict(X, b):
    return np.column_stack([np.ones(len(X)), X]) @ b

# log-HAR
Xl_tr = train[['logRV_d','logRV_w','logRV_m']].values
yl_tr = train['logRV_next'].values
b_log, r2_log_tr = ols_fit(Xl_tr, yl_tr)

Xl_te = test[['logRV_d','logRV_w','logRV_m']].values
pred_log = np.exp(ols_predict(Xl_te, b_log))
actual   = test['RV_next'].values

# GARCH benchmark
def garch_filter(params, r):
    o, a, b = params
    n = len(r); s2 = np.zeros(n); s2[0] = np.var(r)
    for t in range(1, n):
        s2[t] = o + a*r[t-1]**2 + b*s2[t-1]
    return s2

def neg_ll(params, r):
    o, a, b = params
    if o<=0 or a<0 or b<0 or a+b>=1: return 1e10
    s2 = np.maximum(garch_filter(params, r), 1e-12)
    return 0.5*np.sum(np.log(2*np.pi*s2) + r**2/s2)

r_tr = train['RV_d'].values**0.5  # proxy retorno = √RV
s0   = np.var(r_tr)
res_g = minimize(neg_ll, [s0*0.05, 0.08, 0.85], args=(r_tr,),
                 method='L-BFGS-B',
                 bounds=[(1e-8,None),(1e-4,0.5),(1e-4,0.9999)])
o_g, a_g, b_g = res_g.x

r_all = har_df['RV_d'].values**0.5
s2_g  = garch_filter([o_g, a_g, b_g], r_all)
pred_garch = s2_g[split:split+len(actual)]

# Métricas
def r2oos(y, yhat):
    return 1 - ((y-yhat)**2).sum() / ((y-y.mean())**2).sum()
def mae(y, yhat):
    return np.abs(y-yhat).mean()

print('── Parámetros log-HAR ────────────────────────────────────────')
print(f'  β₀={b_log[0]:.4f}  β_d={b_log[1]:.4f}  β_w={b_log[2]:.4f}  β_m={b_log[3]:.4f}')
print(f'  R² in-sample : {r2_log_tr:.4f}')
print(f'  Suma β_d+β_w+β_m = {b_log[1]+b_log[2]+b_log[3]:.4f} (persistencia total)')

print('\n── Métricas OOS ─────────────────────────────────────────────')
print(f'  log-HAR-RV : MAE={mae(actual,pred_log):.6f}  R²OOS={r2oos(actual,pred_log):.4f}')
print(f'  GARCH(1,1) : MAE={mae(actual,pred_garch):.6f}  R²OOS={r2oos(actual,pred_garch):.4f}')
print(f'  Naive RV_t : MAE={mae(actual,test["RV_d"].values):.6f}  R²OOS={r2oos(actual,test["RV_d"].values):.4f}')
print(f'\n── GARCH: α={a_g:.4f}  β={b_g:.4f}  α+β={a_g+b_g:.4f}')

In [ ]:
# ── SAFETY STOCK DINÁMICO ─────────────────────────────────────────────────────
L = 4     # lead time 4 semanas (insumos importados Alicorp)
z = 1.645 # fill rate 95%

# σ condicional = √(RV_forecast)
# RV aquí es varianza de variación de precio → se multiplica por demanda media
# para obtener SS en unidades de demanda
mu_demand = 420  # cajas/sem (del modelo T4)

# SS dinámico: usa RV_forecast como proxy de varianza de demanda relativa
sigma_dyn_price = np.sqrt(pred_log)          # σ de variación de precio
sigma_dyn_demand = sigma_dyn_price * mu_demand  # σ demanda = σ_precio × μ_demanda
ss_dyn   = z * sigma_dyn_demand * np.sqrt(L)

sigma_cls = s.std() * mu_demand
ss_cls    = z * sigma_cls * np.sqrt(L)

har_df_full = har_df.copy()
# Predicciones sobre toda la muestra (in-sample + OOS)
X_all = har_df_full[['logRV_d','logRV_w','logRV_m']].values
pred_all = np.exp(ols_predict(X_all, b_log))
har_df_full['pred_rv']    = pred_all
har_df_full['sigma_dyn']  = np.sqrt(pred_all) * mu_demand
har_df_full['ss_dyn']     = z * har_df_full['sigma_dyn'] * np.sqrt(L)
har_df_full['ss_cls']     = ss_cls

print('── Safety Stock dinámico (HAR) vs. clásico ──────────────────')
print(f'  SS clásico        : {ss_cls:.0f} cajas (fijo, usa σ hist. IPC)')
print(f'  SS HAR μ anual    : {har_df_full.ss_dyn.mean():.0f} cajas')
print(f'  SS HAR en crisis  : {har_df_full.ss_dyn.quantile(0.90):.0f} cajas (p90)')
print(f'  SS HAR en calma   : {har_df_full.ss_dyn.quantile(0.10):.0f} cajas (p10)')
print(f'\n  → En crisis de precios: SS HAR es '
      f'{har_df_full.ss_dyn.quantile(0.90)/ss_cls:.1f}x el clásico')
print(f'  → En calma: SS HAR es '
      f'{har_df_full.ss_dyn.quantile(0.10)/ss_cls:.2f}x el clásico')

In [ ]:
# ── DASHBOARD PRINCIPAL ───────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 13))
fig.suptitle(
    'HAR-RV Safety Stock Dinámico — Alicorp Aceites & Harinas\n'
    f'Basado en volatilidad IPC Alimentos INEI Lima · {SOURCE}',
    fontsize=12, fontweight='bold', y=0.99
)
gs = gridspec.GridSpec(4, 2, hspace=0.40, wspace=0.28)

# P1 — IPC nivel + volatilidad
ax1 = fig.add_subplot(gs[0, :])
ax1_twin = ax1.twinx()
ax1.plot(ipc_weekly.index, ipc_weekly.values, color=C['ipc'], lw=1.0, label='IPC Alimentos')
ax1_twin.fill_between(s.index, np.abs(s)*100, alpha=0.3, color=C['rv'])
ax1_twin.set_ylabel('|Variación semanal| (%)', color=C['rv'])
ax1.set_ylabel('IPC (base 2009=100)', color=C['ipc'])
ax1.set_title('Panel 1 — IPC Alimentos Lima + Volatilidad semanal', loc='left', fontsize=10)
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(axis='y', alpha=0.3)

# P2 — Tres escalas RV
ax2 = fig.add_subplot(gs[1, :])
ax2.semilogy(har_df.index, har_df.RV_d,  color=C['rv'],      lw=0.7,  alpha=0.6, label='RV_d (semanal)')
ax2.semilogy(har_df.index, har_df.RV_w,  color=C['ipc'],     lw=1.0,  label='RV_w (4 semanas)')
ax2.semilogy(har_df.index, har_df.RV_m,  color=C['neutral'], lw=1.2,  label='RV_m (13 semanas)')
ax2.set_ylabel('RV (log scale)')
ax2.set_title('Panel 2 — Varianza Realizada a 3 escalas (d=1, w=4, m=13 semanas)', loc='left', fontsize=10)
ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

# P3 — Forecast HAR vs. GARCH vs. real (OOS)
ax3 = fig.add_subplot(gs[2, :])
test_dates = test.index
n_te = min(len(test_dates), len(actual), len(pred_log), len(pred_garch))
ax3.semilogy(test_dates[:n_te], actual[:n_te],       color=C['rv'],    lw=0.8,       label='RV real')
ax3.semilogy(test_dates[:n_te], pred_log[:n_te],     color=C['har'],   lw=1.3,       label=f'log-HAR  R²={r2oos(actual,pred_log):.3f}')
ax3.semilogy(test_dates[:n_te], pred_garch[:n_te],   color=C['garch'], lw=1.3, ls='--', label=f'GARCH    R²={r2oos(actual,pred_garch):.3f}')
ax3.set_ylabel('RV forecast (log scale)')
ax3.set_title('Panel 3 — Forecast OOS: log-HAR vs. GARCH (20% test)', loc='left', fontsize=10)
ax3.legend(fontsize=8); ax3.grid(axis='y', alpha=0.3)

# P4 — SS dinámico vs. clásico
ax4 = fig.add_subplot(gs[3, 0])
ax4.fill_between(har_df_full.index, har_df_full.ss_dyn, har_df_full.ss_cls,
                 where=har_df_full.ss_dyn >= har_df_full.ss_cls,
                 alpha=0.35, color=C['rv'], label='SS HAR > clásico (alerta precio)')
ax4.fill_between(har_df_full.index, har_df_full.ss_dyn, har_df_full.ss_cls,
                 where=har_df_full.ss_dyn <  har_df_full.ss_cls,
                 alpha=0.35, color=C['ipc'], label='SS HAR < clásico (mercado calmo)')
ax4.plot(har_df_full.index, har_df_full.ss_dyn, color=C['rv'],      lw=0.9)
ax4.plot(har_df_full.index, har_df_full.ss_cls, color=C['neutral'],  lw=1.2, ls='--')
ax4.set_ylabel('SS (cajas)')
ax4.set_title('Panel 4 — SS HAR vs. SS clásico\n(Alicorp, lead time 4 sem)', loc='left', fontsize=9)
ax4.legend(fontsize=7); ax4.grid(axis='y', alpha=0.3)

# P5 — Coeficientes HAR
ax5 = fig.add_subplot(gs[3, 1])
nombres = ['β₀', 'β_d\n(semanal)', 'β_w\n(mensual)', 'β_m\n(trimestral)']
colores = [C['neutral'], C['rv'], C['ipc'], C['neutral']]
bars = ax5.bar(nombres, b_log, color=colores, alpha=0.85, edgecolor='white')
ax5.axhline(0, color='black', lw=0.5)
for bar, val in zip(bars, b_log):
    offset = 0.01 if val >= 0 else -0.03
    ax5.text(bar.get_x() + bar.get_width()/2, val + offset,
             f'{val:.3f}', ha='center', fontsize=9)
ax5.set_title('Panel 5 — Coeficientes log-HAR\n(heterogeneidad de horizonte)', loc='left', fontsize=9)
ax5.set_ylabel('Coeficiente'); ax5.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/supply_dashboard.png', dpi=140, bbox_inches='tight')
plt.show()
print('✓ data/supply_dashboard.png')

In [ ]:
# ── FORECAST SS PRÓXIMAS 4 SEMANAS ───────────────────────────────────────────
last = har_df.iloc[-1]
print('── Forecast Safety Stock — próximas 4 semanas ───────────────')
print(f'  Estado actual: RV_d={last.RV_d:.6f}  RV_w={last.RV_w:.6f}  RV_m={last.RV_m:.6f}')
print()

rv_d_cur = last.RV_d
rv_w_cur = last.RV_w
rv_m_cur = last.RV_m

for h in range(1, 5):
    log_pred = (b_log[0]
                + b_log[1]*np.log(max(rv_d_cur, 1e-12))
                + b_log[2]*np.log(max(rv_w_cur, 1e-12))
                + b_log[3]*np.log(max(rv_m_cur, 1e-12)))
    rv_forecast = np.exp(log_pred)
    sigma_f     = np.sqrt(rv_forecast) * mu_demand
    ss_f        = z * sigma_f * np.sqrt(L)
    print(f'  h={h} (sem +{h}): RV_f={rv_forecast:.6f}  σ={sigma_f:.1f}  SS={ss_f:.0f} cajas '
          f'(vs. clásico {ss_cls:.0f})')
    # Actualizar para siguiente iteración
    rv_d_cur = rv_forecast
    rv_w_cur = (rv_w_cur * 3 + rv_forecast) / 4
    rv_m_cur = (rv_m_cur * 12 + rv_forecast) / 13

In [ ]:
# ── EXPORTAR ─────────────────────────────────────────────────────────────────
har_df_full.to_csv('data/supply_har_output.csv')
print('✓ data/supply_har_output.csv')
print('✓ data/supply_eda.png')
print('✓ data/supply_dashboard.png')

## Conclusiones

| Concepto | Finanzas (S&P 500) | Supply Chain (Alicorp / INEI) |
|----------|--------------------|-------------------------------|
| **RV** | Suma r² intradiarios (78 obs/día) | Variación² IPC semanal (proxy) |
| **RV_d** | Volatilidad del día | Varianza del error esta semana |
| **RV_w** | Avg 5 días | Avg 4 semanas |
| **RV_m** | Avg 22 días | Avg 13 semanas (1 trimestre) |
| **β_d > β_w > β_m** | HFT > gestores > directores riesgo | Reacción rápida > mensual > trimestral |
| **log-HAR** | Mejor R² OOS | Preferido por simetría de residuos |
| **SS HAR** | Colateral FX dinámico | SS sube ante volatilidad IPC alta |

**Ventaja clave sobre GARCH para Supply:** el componente $\beta_m \cdot RV_m^{(m)}$ actúa como **señal de alerta temprana** de presión inflacionaria estructural — cuando el IPC lleva 13 semanas con variabilidad elevada (como la crisis de aceites 2022), el SS HAR ya incorpora esa señal aunque la última semana haya sido tranquila. El GARCH con su memoria exponencial la subestimaría.

**Limitación:** la RV construida con variaciones semanales es solo un proxy de la varianza verdadera. Con datos diarios de precios mayoristas (Minagri) se obtiene una RV de mayor calidad (7 obs/semana en vez de 1).

**Conexión con T6 (Hurst):** la persistencia larga de la RV del IPC sugiere que el proceso puede ser fractalmente integrado ($d > 0$). El exponente de Hurst cuantifica exactamente esa persistencia — si $H > 0.5$ confirma long memory y justifica modelos ARFIMA sobre HAR.